# RSP Experiment Notebook — Cao et al. (2020) Reproduction

Run cells in order. Modify parameters in **§2 Configuration** to switch datasets or tune experiment settings.

Corresponds to `run.py` workflow, broken into inspectable steps.

## 1. Setup — Imports & Paths

In [ ]:
import gc
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Ensure src/ is importable
PROJECT_ROOT = Path().resolve()  # or Path("./Cao_SOTA_MP").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph import RoadNetwork
from src.generator import compute_deadline
from src.ilp_solver import solve_ilp
from src.milp_solver import solve_milp
from src.dijkstra_solver import solve_dijkstra
from src.experiment import load_experiment_data, _compute_path_stats, _path_match
from src.visualize import (
    plot_accuracy_vs_deadline,
    plot_probability_comparison,
    print_summary,
)

print(f"Project root: {PROJECT_ROOT}")
print("Imports OK.")

## 2. Configuration

Edit the dict below to switch datasets or tune parameters.

In [ ]:
# ============================================================
# Change these to match what you want to run
# ============================================================

# Option A: load pre-generated data (recommended)
# DATA_DIR = "data/full/seed42"  # artificial 65-node
# DATA_DIR = "data/beijing"             # Beijing OSM 587-node
DATA_DIR = "data/small"  # 10-node debug

# Option B: inline generation (set to None, uses config below)
# DATA_DIR = None

# Solver settings
BACKEND = "SCIP"  # SCIP / CBC / GLPK
BIG_M = 1_000_000  # big-M cap (per-sample M bounded by this)
TIME_LIMIT = 60  # solver time limit (seconds)

# Experiment settings (used if DATA_DIR is None, or for overriding num_repeats)
ALPHAS = [0.5, 0.6, 0.7, 0.8, 0.9]
MAX_CANDIDATE_PATHS = 1000

# Output
OUTPUT_DIR = "results/"
SAVE_CSV = True
GENERATE_PLOTS = True

# ============================================================

print(f"Data:      {DATA_DIR or '(inline generation)'}")
print(f"Backend:   {BACKEND}")
print(f"Alphas:    {ALPHAS}")
print(f"Output:    {OUTPUT_DIR}")

## 3. Load (or Generate) Data

Loads `network.npz`, `travel_times.npz`, `od_pairs.npy`, `meta.yaml` from `DATA_DIR`.

In [ ]:
if DATA_DIR:
    data = load_experiment_data(DATA_DIR)
    network = data["network"]
    od_pairs_all = data["od_pairs"]
    travel_times_all = data["travel_times"]
    meta = data["meta"]
    print(f"Loaded: {network.num_nodes} nodes, {network.num_edges} edges")
    print(f"  OD pairs: {len(od_pairs_all)}")
    print(f"  Repeats:  {len(travel_times_all)}")
    print(f"  Samples:  {travel_times_all[0].shape[0]}")
    print(
        f"  Meta:     { {k: v for k, v in meta.items() if k not in ('travel_time_model',)} }"
    )

    # Control how many to run
    NUM_REPEATS = min(3, len(travel_times_all))  # reduce for faster runs
    NUM_OD = min(5, len(od_pairs_all))  # reduce for faster runs
    od_pairs = od_pairs_all[:NUM_OD]
    travel_times = travel_times_all[:NUM_REPEATS]

    # Baseline seed (used in compute_deadline for per-sample SP sampling)
    BASE_SEED = meta.get("seed", 42)
else:
    # Inline generation (legacy, from config)
    from src.generator import (
        create_artificial_network,
        generate_travel_times,
        random_od_pairs,
    )

    network = create_artificial_network(65, 123, seed=42)
    od_pairs = random_od_pairs(network, 20, seed=42)
    travel_times = [
        generate_travel_times(network.num_edges, 500, (10.0, 100.0), seed=1000 + r)
        for r in range(3)
    ]
    NUM_REPEATS = len(travel_times)
    NUM_OD = len(od_pairs)
    BASE_SEED = 42

total_jobs = NUM_REPEATS * NUM_OD * len(ALPHAS)
print(
    f"\nWill run: {NUM_REPEATS} repeats × {NUM_OD} OD pairs × {len(ALPHAS)} α = {total_jobs} jobs"
)

### 3a. Inspect Network & Data

In [ ]:
# Quick look at the data
print(f"Nodes: {network.num_nodes}, Edges: {network.num_edges}")
print(f"OD pairs sample: {od_pairs[:3]}")
print(f"W shape (repeat 0): {travel_times[0].shape}")

# Edge stats for repeat 0
W0 = travel_times[0]
edge_means = W0.mean(axis=0)
edge_cvs = W0.std(axis=0) / edge_means
print(
    f"\nEdge travel time — mean: [{edge_means.min():.1f}, {edge_means.max():.1f}] min"
)
print(
    f"Edge CV               — median: {np.median(edge_cvs):.3f}, range: [{edge_cvs.min():.3f}, {edge_cvs.max():.3f}]"
)

## 4. Run a Single Job (Debug/Explore)

Tweak `REPEAT`, `OD_IDX`, `ALPHA` to run one specific case and inspect solver outputs.

In [ ]:
# ============================================================
# Change these to debug a specific case
REPEAT = 0  # which repeat
OD_IDX = 0  # which OD pair
ALPHA = 0.5  # deadline level
# ============================================================

W = travel_times[REPEAT]
N = W.shape[0]
o, d = od_pairs[OD_IDX]

print(f"Repeat={REPEAT}, OD=({o}→{d}), α={ALPHA}")
print(f"W shape: {W.shape}")
print()

# Compute deadline
t0 = time.perf_counter()
tau, tau_diag = compute_deadline(
    W,
    network,
    o,
    d,
    ALPHA,
    max_candidate_paths=MAX_CANDIDATE_PATHS,
    seed=BASE_SEED,
    return_diagnostics=True,
)
print(
    f"Deadline τ = {tau:.1f}  (T_min={tau_diag['T_min']:.1f}, T_max={tau_diag['T_max']:.1f}, candidates={tau_diag['candidate_count']})"
)

# Run ILP (ground-truth)
t0 = time.perf_counter()
ilp = solve_ilp(
    network, W, o, d, tau, big_m=BIG_M, solver_name=BACKEND, time_limit=TIME_LIMIT
)
t_ilp = time.perf_counter() - t0

# Run MILP
t0 = time.perf_counter()
milp = solve_milp(network, W, o, d, tau, solver_name=BACKEND, time_limit=TIME_LIMIT)
t_milp = time.perf_counter() - t0

# Run Dijkstra
t0 = time.perf_counter()
dij = solve_dijkstra(network, W, o, d, tau)
t_dij = time.perf_counter() - t0

# Report
print()
for name, r, t in [
    ("ILP", ilp, t_ilp),
    ("MILP", milp, t_milp),
    ("Dijkstra", dij, t_dij),
]:
    status = r["status"]
    prob = r["punctuality_prob"]
    late = r.get("lateness_count", "?")
    print(
        f"  {name:10s}: status={status:8s}  punct={prob:.4f}  late={late}/{N}  time={t:.3f}s"
    )

### 4a. Inspect Path Details

In [ ]:
# Which edges were chosen?
ilp_edges = [
    (network.edges[j][0], network.edges[j][1]) for j in np.where(ilp["path_x"] > 0.5)[0]
]
dij_edges = [
    (network.edges[j][0], network.edges[j][1]) for j in np.where(dij["path_x"] > 0.5)[0]
]

print(f"ILP path ({len(ilp_edges)} edges): {ilp_edges}")
print(f"Dij path ({len(dij_edges)} edges): {dij_edges}")
print(f"Same path: {np.array_equal(ilp['path_x'], dij['path_x'])}")

# Path travel time distributions
ilp_times = W @ ilp["path_x"]
dij_times = W @ dij["path_x"]
print(
    f"\nILP path times — mean={ilp_times.mean():.1f}, max={ilp_times.max():.1f}, late={(ilp_times > tau).sum()}/{N}"
)
print(
    f"Dij path times — mean={dij_times.mean():.1f}, max={dij_times.max():.1f}, late={(dij_times > tau).sum()}/{N}"
)

## 5. Run Full Experiment

Run all (repeat × OD × α) combinations. Progress printed per job.

In [ ]:
results = []
deadline_diag_by_alpha = {a: [] for a in ALPHAS}

t_start = time.perf_counter()
job_idx = 0

for repeat in range(NUM_REPEATS):
    W = travel_times[repeat]
    N = W.shape[0]
    t_repeat = time.perf_counter()

    for oi, (o, d) in enumerate(od_pairs):
        for ai, alpha in enumerate(ALPHAS):
            job_idx += 1

            tau, tau_diag = compute_deadline(
                W,
                network,
                o,
                d,
                alpha,
                max_candidate_paths=MAX_CANDIDATE_PATHS,
                seed=BASE_SEED + repeat * 100 + oi,
                return_diagnostics=True,
            )
            deadline_diag_by_alpha[alpha].append(tau_diag)

            # ILP
            t0 = time.perf_counter()
            ilp_result = solve_ilp(network, W, o, d, tau, BIG_M, BACKEND, TIME_LIMIT)
            t_ilp = time.perf_counter() - t0

            if ilp_result["status"] != "Optimal":
                print(
                    f"  [{job_idx}/{total_jobs}] R{repeat + 1} OD{oi + 1} α={alpha} ILP={ilp_result['status']} ⚠ SKIP"
                )
                continue

            ilp_path = ilp_result["path_x"]
            ilp_prob = ilp_result["punctuality_prob"]
            ilp_stats = _compute_path_stats(W, ilp_path, tau)

            # MILP
            t0 = time.perf_counter()
            milp_result = solve_milp(network, W, o, d, tau, BACKEND, TIME_LIMIT)
            t_milp = time.perf_counter() - t0
            milp_path = (
                milp_result["path_x"] if milp_result["status"] == "Optimal" else None
            )
            milp_prob = (
                milp_result["punctuality_prob"]
                if milp_result["status"] == "Optimal"
                else None
            )

            # Dijkstra
            t0 = time.perf_counter()
            dij_result = solve_dijkstra(network, W, o, d, tau)
            t_dij = time.perf_counter() - t0
            dij_path = dij_result["path_x"]
            dij_prob = dij_result["punctuality_prob"]

            # Gaps & tie-aware
            gap_milp = (
                ilp_prob - milp_prob
                if (ilp_prob is not None and milp_prob is not None)
                else None
            )
            milp_tie_ok = abs(gap_milp) <= 1.0 / N if gap_milp is not None else None
            gap_dij = (
                ilp_prob - dij_prob
                if (ilp_prob is not None and dij_prob is not None)
                else None
            )
            dij_tie_ok = abs(gap_dij) <= 1.0 / N if gap_dij is not None else None

            milp_stats = _compute_path_stats(W, milp_path, tau)
            dij_stats = _compute_path_stats(W, dij_path, tau)

            # Collect results
            for method, prob, path_match, gap, tie_ok, stats, solve_t in [
                ("ILP", ilp_prob, True, 0.0, True, ilp_stats, ilp_result["solve_time"]),
                (
                    "MILP",
                    milp_prob,
                    _path_match(milp_path, ilp_path),
                    gap_milp,
                    milp_tie_ok,
                    milp_stats,
                    milp_result["solve_time"],
                ),
                (
                    "Dijkstra",
                    dij_prob,
                    _path_match(dij_path, ilp_path),
                    gap_dij,
                    dij_tie_ok,
                    dij_stats,
                    dij_result["solve_time"],
                ),
            ]:
                results.append(
                    {
                        "repeat": repeat,
                        "od_idx": oi,
                        "origin": o,
                        "dest": d,
                        "alpha": alpha,
                        "method": method,
                        "punctuality_prob": prob,
                        "correct": path_match,
                        "objective_gap": gap,
                        "tie_aware_correct": tie_ok,
                        "late_count": stats["late_count"],
                        "delay_sum": stats["delay_sum"],
                        "max_delay": stats["max_delay"],
                        "path_length": stats["path_length"],
                        "mean_time": stats["mean_time"],
                        "solve_time": solve_t,
                    }
                )

            # Progress
            elapsed = time.perf_counter() - t_start
            avg_per_job = elapsed / job_idx
            eta = avg_per_job * (total_jobs - job_idx)
            eta_str = (
                f"{eta / 60:.0f}m{eta % 60:.0f}s"
                if eta < 3600
                else f"{eta / 3600:.1f}h"
            )
            print(
                f"  [{job_idx}/{total_jobs}] R{repeat + 1} OD{oi + 1} α={alpha} | "
                f"ILP={t_ilp:.1f}s MILP={t_milp:.1f}s Dij={t_dij:.3f}s | "
                f"elapsed={elapsed / 60:.1f}m ETA={eta_str}"
            )

    t_repeat_elapsed = time.perf_counter() - t_repeat
    total_elapsed = time.perf_counter() - t_start
    print(
        f"  Repeat {repeat + 1}/{NUM_REPEATS} done ({t_repeat_elapsed / 60:.1f}m, total {total_elapsed / 60:.1f}m)"
    )
    gc.collect()

df = pd.DataFrame(results)
print(f"\nDone! {len(df)} rows ({len(df) // 3} jobs × 3 methods)")

## 6. Results Summary

In [ ]:
print_summary(df)

### 6a. Accuracy by α (Table)

In [ ]:
# Path-match accuracy by alpha and method
acc_by_alpha = df.pivot_table(
    index="alpha", columns="method", values="correct", aggfunc="mean"
)
display(acc_by_alpha.style.format("{:.1%}").set_caption("Path-Match Accuracy by α"))

# Tie-aware accuracy by alpha and method
tie_by_alpha = df.pivot_table(
    index="alpha", columns="method", values="tie_aware_correct", aggfunc="mean"
)
display(tie_by_alpha.style.format("{:.1%}").set_caption("Tie-Aware Accuracy by α"))

### 6b. Solve Time Statistics

In [ ]:
time_stats = df.groupby("method")["solve_time"].agg(["mean", "median", "max", "std"])
display(time_stats.style.format("{:.4f}").set_caption("Solve Time Statistics (s)"))

### 6c. Objective Gap Distribution

In [ ]:
gap_data = df[df["method"] != "ILP"].dropna(subset=["objective_gap"])
print("Objective gap (p_ILP - p_method):")
for method in ["MILP", "Dijkstra"]:
    g = gap_data[gap_data["method"] == method]["objective_gap"]
    if len(g) > 0:
        print(
            f"  {method:10s}: mean={g.mean():.5f}  max={g.max():.5f}  min={g.min():.5f}"
        )

### 6d. Deadline Diagnostics

In [ ]:
print("=== Deadline Diagnostics ===")
for alpha in ALPHAS:
    diags = deadline_diag_by_alpha[alpha]
    ilp_rows = df[(df["method"] == "ILP") & (df["alpha"] == alpha)]
    ilp_puncts = ilp_rows["punctuality_prob"].dropna()
    all_cand_puncts = []
    for d in diags:
        all_cand_puncts.extend(d.get("candidate_puncts", []))
    cand_mean = np.mean(all_cand_puncts) if all_cand_puncts else float("nan")
    print(
        f"  α={alpha}: tau mean={np.mean([d['tau'] for d in diags]):.1f} "
        f"ILP_punct median={ilp_puncts.median():.3f} mean={ilp_puncts.mean():.3f} "
        f"min={ilp_puncts.min():.3f} max={ilp_puncts.max():.3f} | "
        f"cand_punct mean={cand_mean:.3f}"
    )

## 7. Plots

In [ ]:
if GENERATE_PLOTS:
    plot_accuracy_vs_deadline(df, f"{OUTPUT_DIR}/figures")
    plot_probability_comparison(df, f"{OUTPUT_DIR}/figures")

### 7a. Inline Plots (Optional)

Display plots directly in notebook.

In [ ]:
import matplotlib.pyplot as plt

# Accuracy vs alpha
acc = df.groupby(["alpha", "method"])["correct"].mean().reset_index()
acc.columns = ["alpha", "method", "accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
for method in ["ILP", "MILP", "Dijkstra"]:
    subset = acc[acc["method"] == method]
    ax.plot(subset["alpha"], subset["accuracy"], "o-", label=method, markersize=8)
ax.set_xlabel("α (deadline level)")
ax.set_ylabel("Path-Match Accuracy")
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title("Accuracy vs Deadline")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Tie-aware accuracy vs alpha
tie_acc = df.groupby(["alpha", "method"])["tie_aware_correct"].mean().reset_index()
tie_acc.columns = ["alpha", "method", "tie_accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
for method in ["MILP", "Dijkstra"]:
    subset = tie_acc[tie_acc["method"] == method]
    ax.plot(subset["alpha"], subset["tie_accuracy"], "s--", label=method, markersize=8)
ax.set_xlabel("α (deadline level)")
ax.set_ylabel("Tie-Aware Accuracy")
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title("Tie-Aware Accuracy vs Deadline (|gap| ≤ 1/N)")
ax.grid(True, alpha=0.3)
plt.show()

## 8. Save Outputs

In [ ]:
from src.experiment import _save_worst_cases

if SAVE_CSV:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    csv_path = f"{OUTPUT_DIR}/results.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    _save_worst_cases(df, OUTPUT_DIR)

print("\nNotebook done.")